In [113]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
from torchvision import models
import ssl

from glob import glob
from PIL import Image
import random
import os
import pandas as pd
import json
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler
from concurrent.futures import ThreadPoolExecutor
import pickle

## kiểm tra dữ liệu

In [114]:
df1 = pd.read_csv('data/csv/cicddos_2019_6_labels.csv')
# df2 = pd.read_csv('temp/actual_data/test_syn.csv')
df2 = pd.read_csv('temp/actual_data/test_udp.csv')

In [115]:
selected_columns1 = ['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 
                    'Fwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 
                    'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 
                    'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 
                    'Flow IAT Max', 'Flow IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Min', 
                    'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 
                    'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Max', 'FIN Flag Count', 'SYN Flag Count', 'Label']

df1 = df1[df1['Label'] == 'UDP-Combined']
df1 = df1[selected_columns1]

labels = df1['Label']




In [116]:
labels.value_counts()

Label
UDP-Combined    5000
Name: count, dtype: int64

In [125]:
df1.sample(5)

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Bwd Packets/s,Packet Length Max,FIN Flag Count,SYN Flag Count,Label
26427,17,1,2,0,766.0,383.0,383.0,0.0,0.0,0.0,...,0,0,0,0,0,0.0,383.0,0,0,UDP-Combined
25297,17,49,2,0,766.0,383.0,383.0,0.0,0.0,0.0,...,0,0,0,40,0,0.0,383.0,0,0,UDP-Combined
28023,6,1,2,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,40,0,0.0,0.0,0,0,UDP-Combined
29711,6,1,2,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,40,0,0.0,0.0,0,0,UDP-Combined
28747,17,3,2,0,2944.0,1472.0,1472.0,0.0,0.0,0.0,...,0,0,0,64,0,0.0,1472.0,0,0,UDP-Combined


In [118]:
selected_columns2 = ['Source IP','Dest IP','Protocol','Flow Duration', 'Total Fwd Packets', 
                     'Total Backward Packets', 'Fwd Packets Length Total', 'Fwd Packet Length Max', 
                     'Fwd Packet Length Min', 'Bwd Packets Length Total']

df2 = df2[selected_columns2]

In [119]:
df2.sample(10)

,Source IP,Dest IP,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Bwd Packets Length Total
21,192.168.1.56,192.168.200.60,17,45345.349312,23,0,32384,1472.0,0.0,0
8,192.168.1.64,192.168.200.60,17,48960.716724,39,0,54600,1400.0,1400.0,0
91,192.168.1.44,192.168.200.60,17,48334.250450,3,0,3444,1148.0,1148.0,0
93,192.168.1.84,192.168.200.60,17,46846.711636,3,0,3066,1022.0,1022.0,0
86,192.168.1.33,192.168.200.60,17,27954.461575,23,0,31303,1361.0,1361.0,0
84,192.168.1.36,192.168.200.60,17,34601.976871,23,0,14536,632.0,632.0,0
45,192.168.1.23,192.168.200.60,17,46182.186604,25,0,19500,780.0,780.0,0
142,192.168.1.71,192.168.200.60,17,35510.246754,23,0,11247,489.0,489.0,0
68,192.168.1.39,192.168.200.60,17,41155.295372,24,0,33408,1392.0,1392.0,0
122,192.168.1.56,192.168.200.60,17,990.447998,4,0,5888,1472.0,1472.0,0


## chuyên dữ liệu

#### chuyển dữ liệu cách 2

In [120]:
selected_columns = ['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 
                    'Fwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 
                    'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 
                    'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 
                    'Flow IAT Max', 'Flow IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Min', 
                    'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 
                    'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Max', 'FIN Flag Count', 'SYN Flag Count', 
                    'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 
                    'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 
                    'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 
                    'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Std', 'Label']


# Tải scaler đã lưu
with open("minmax_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

def process_task(task):
    row, label, img_number, output_base = task
    num_features = row.shape[0]
    s = int(np.ceil(np.sqrt(num_features)))
    padded_size = s * s
    padded_row = np.zeros(padded_size)
    padded_row[:num_features] = row
    image_2d = padded_row.reshape((s, s))

    size = 224
    upscale_factor= size // s 
    expanded_image = np.kron(image_2d, np.ones((upscale_factor, upscale_factor)))

    scaled_image = (expanded_image * 255).astype(np.uint8)

    img = Image.fromarray(scaled_image, mode='L')
    img = img.convert("RGB")
    img = img.resize((size, size))
    img.save(os.path.join(output_base, str(label), f'{img_number}.png'))

def deepinsight_conversion(input_csv, output_dir='output_images', max_per_label=200, workers=3):
    # Đọc dữ liệu
    df = pd.read_csv(input_csv)
    df = df[selected_columns]

    labels = df['Label']
    data = df.drop(columns=['Label'])

    data.replace([-np.inf, np.inf], 0, inplace=True)
    data.fillna(0, inplace=True)  # Điền giá trị NaN bằng giá trị trung vị
    std = data.std()
    std.replace(0, 1, inplace=True)
    data = np.log1p(data + 1)
    data.replace([-np.inf, np.inf], 0, inplace=True)
    data.fillna(0, inplace=True)  # Điền giá trị NaN bằng giá trị trung vị

    features = data.values

    # Chuẩn hóa dữ liệu
    normalized_features = scaler.fit_transform(features)
    # normalized_features = features

    # Tạo thư mục đầu ra
    os.makedirs(output_dir, exist_ok=True)
    unique_labels = pd.unique(labels)
    for label in unique_labels:
        os.makedirs(os.path.join(output_dir, str(label)), exist_ok=True)

    # Chuẩn bị tasks
    tasks = []
    for label in unique_labels:
        mask = (labels == label).values
        label_features = normalized_features[mask][:max_per_label]
        for img_number, row in enumerate(label_features):
            tasks.append((row, label, img_number, output_dir))

    # Xử lý song song
    with ThreadPoolExecutor(max_workers=workers) as executor:
        executor.map(process_task, tasks)



# Sử dụng hàm
deepinsight_conversion(
    input_csv='temp/actual_data/test_udp.csv',
    output_dir='temp/data',
    max_per_label=200,
    workers=3
)



## Danh gia

In [12]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

In [ ]:
label_map = {'BENIGN': 0, 'LDAP': 1, 'MSSQL': 2, 'NetBIOS-Portmap': 3, 'Syn': 4, 'UDP-Combined': 5}

In [14]:
device = torch.device("cpu")
print("Using device:", device)

Using device: cpu


In [15]:
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(label_map.keys()))  # Adjust output layer
model = model.to(device)
model.load_state_dict(torch.load("temp/models/resnet50_finetuned.pth",  map_location=torch.device('cpu')))
print(len(label_map.keys()))

num_classes = len(label_map.keys())

6


In [16]:
# Test individual images
def predict_image(model, image_path, class_names):
    model.eval()
    image = Image.open(image_path)
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    image = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(image)
        _, predicted = torch.max(output, 1)
    # print(f'Predicted label: {class_names[predicted.item()]}')
    return class_names[predicted.item()]



In [104]:
predict_image(model, "temp/images/Syn/82.png", list(label_map.keys()))

'Syn'

In [192]:
folder_path = "temp/images/Syn"
files = os.listdir(folder_path)  # Lấy danh sách tất cả file và thư mục
name_details_dict = {}

for file in files:
    r = predict_image(model, f"temp/images/Syn/{file}", list(label_map.keys()))
    if r in name_details_dict:
        name_details_dict[r] += 1
    else:
        name_details_dict[r] = 1
print(name_details_dict)

{'BENIGN': 64, 'Syn': 7, 'UDP-Combined': 2}
